In [1]:
import numpy as np
import math
import graphviz
import random

In [2]:
from graphviz import Digraph

# root is the argument. In this case, it's d
def trace(root):

    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()

    # v is a Value object that we defined above!
    def build(v):
        
        if v not in nodes:
            nodes.add(v)

            for child in v._prev:

                # add a tuple of (child, v) to the edges array (add an edge from the child to the v (result) node)
                edges.add((child, v))

                # recursively build each child (and their children) as its own node
                build(child)            

    build(root)
    return nodes, edges

def draw_dot(root):
    
    # create a graph named dot
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = Left to Right

    # run trace to generate nodes and edges tuples
    nodes, edges = trace(root)


    for n in nodes:

        # create a unique id for each node
        uid = str(id(n))

        # for any value in the graph, create a rectangluar ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')

        # if this value is the result of some operation, create an op node for it
        if n._op:
            dot.node(name = uid + n._op, label = n._op)

            # and connect this node to it. Edge follows left-> right because of the argument we passed to graph creation earlier
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2. The first execution of this is creating an edge between c's node and d's operation.
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot



In [3]:
class Value:

    def __init__(self, data, _children = (), _op = "", label = ""):
        self.data = data
        self._prev = set(_children) # just for the graph
        self._op = _op
        self.label = label
        
        # stores specific function to calculate gradient of children
        self._backward = lambda: None
        
        # stores gradient
        self.grad = 0
    
    
    # gets called when just the variable name is called
    def __repr__(self):
        return f"Value(data={self.data})"


    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) # converts any other numbers used in arithmetic with a value object into values themselves

        out = Value(self.data + other.data, (self, other), "+")

        # for a+b, when differentiating with respect to a, the derivative is always 1
        
        # when we call _backward on c, we want a and b's derivatives to become 1, then multiply them by out's grad for the chain rule
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward   # make sure it's not _backward(), which calls the function and stores its output in out._backwarwd

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other) # converts any other numbers used in arithmetic with a value object into values themselves

        out = Value(self.data * other.data, (self, other), "*")

        # the derivative of a*b with respect to a is b (the scalar multiple)
        def _backward():
            self.grad += other.data * out.grad # multiply by outgoing gradient for chain rule. Remember! Chain rule is dL/da = da/dc * dc/dL
            other.grad += self.data * out.grad
            
        out._backward = _backward
        
        return out


    # reverse functions

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other
    
    def __rsub__(self, other):
        return (-self) + other
    

    # inverse
    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    # power
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data**(other - 1)) * out.grad
            # other.grad = (self.data**other.data) * math.log(other.data)

        out._backward = _backward

        return out
    
    # division
    def __truediv__(self, other):
        return self * other**-1

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        t = math.exp(x)
        out = Value(t, (self,), 'exp')

        def _backward():
            self.grad += t * out.grad

        out._backward = _backward
        return out
    
    # topological sort

    # we want to backpropagate from the end to the beginning
    # we achieve this by recursively arranging the children from beginning to end

    def backward(self):
        
        topo = []
        visited = set()

        def build_graph(v):
            if v not in visited:
                
                # add child so that it doesn't get added twice
                visited.add(v)
                
                # recursively call on the child to build its children
                for child in v._prev:
                    build_graph(child)
                
                # add self last
                topo.append(v)
        

        # actually call the function to build out the topo graph
        build_graph(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()



In [ ]:
# time to build the neural network!

# neurons, layers, MLP


# neurons take inputs, squash them through an activation function, and return that output
# layers take inputs (from the neurons) and number of outputs as an argument, where nout is the number of neurons in the layer. It then returns that outptut as a list of the neuron's outputs.

# the MLP takes nins (number of inputs to the network), adds that to the array of layers (nouts), and passes each input through each layer, which passes each input through each neuron. This computes the forward pass.


# in this neural net, all neurons in a layer will receive the same number of inputs, but run their own # of transformations and each output one output.
class Neuron:
    
    def __init__(self, nin):
        # set weights and biases
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))


    # x is a list of inputs
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        # send weights and biases upstream
        return self.w + [self.b]
    



# layers store groups of neurons; nin describes the number of inputs going into the layer, nout describes the number of outputs/how many neurons are in that layer
# class Layer(nin, nout):

class Layer:

    def __init__(self, nin, nout):
        
        # for each neuron in nout (number of neurons in network), create a new neuron initialized with the # of inputs going into that layer.
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
        
    # when a layer gets called with the input array, it should send the input to each neuron and record their outputs in a list.
    def __call__(self, x):
        
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

        
        return outs

    # create a list containing all the weights and biases of the layer's neurons and send it upstream
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

            
        
class MLP:

    # nouts is an array containing the input amount for each layer, nin is # of inputs to network.
    # Because each nout contains the next nout's inputs, we store them all in a larger list nouts and iterate over them.
    def __init__(self, nin, nouts):
        

        self.layers = []

        # By adding nin to nouts, we create the entire network's inputs, then iterate over them to build the network.
        net = [nin] + nouts

        # create each layer using the previous' nout as the current nin
        for i in range(len(nouts)):

            # initializes all the layers (neurons now have weights and biases)
            self.layers.append(Layer(net[i], net[i + 1]))


    # when we call the MLP, we want it to evaluate all the layers with the previous
    # layer's output
    def __call__(self, x):

        
        # initialize each layer with the previous layer's output (x)
        # neat way too recursively iterate
        for layer in self.layers:
            x = layer(x)

        # outputs the outputs of the last layer
        return x

    def parameters(self):

        # for layers in self.layers, for p in layers.parameters(), return p
        return [p for layers in self.layers for p in layers.parameters()]

In [5]:
# inputs
x = [2.0, 3.0, -1.0]

# passes number of ins to the network, and defines 3 layers with 4 neurons each. The neurons initialize with random biases and weights.
n = MLP(3, [4, 4, 1])

# the MLP is then called with the inputs, which then call the layers with the inputs sequentially
n(x)
n.parameters()


[Value(data=-0.3915799510881903),
 Value(data=-0.18624743061633997),
 Value(data=0.7485809159046426),
 Value(data=0.4977706958832899),
 Value(data=-0.28540435868850733),
 Value(data=-0.4246652164511835),
 Value(data=-0.5328739711705426),
 Value(data=-0.3298451382201071),
 Value(data=0.7709635952455198),
 Value(data=0.16858831391792162),
 Value(data=-0.8809472629264481),
 Value(data=0.8147960752659711),
 Value(data=-0.30197899356967595),
 Value(data=-0.9358064733197637),
 Value(data=-0.3080160960354925),
 Value(data=0.7957149566227513),
 Value(data=0.2226455327807535),
 Value(data=-0.026290175170500696),
 Value(data=0.9160324856286779),
 Value(data=-0.5776590439881715),
 Value(data=-0.8710564207030498),
 Value(data=-0.577641541765104),
 Value(data=-0.5071811114123932),
 Value(data=-0.23815277643496802),
 Value(data=-0.47613443437019565),
 Value(data=-0.347768358401505),
 Value(data=-0.3254113468875268),
 Value(data=-0.3663606357446054),
 Value(data=0.33713297313320223),
 Value(data=0.06

In [6]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] # desired targets


In [12]:
for k in range(20):

    # forward pass
    ypred = [n(x) for x in xs] # call the MLP using the xs we just defined
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # backward pass
    for p in n.parameters(): # reset gradient of each parameter to 0 so gradients don't get infinitely added
        p.grad = 0.0

    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.1 * p.grad
    
    print(k, loss.data)

0 7.155220833283989
1 2.8314201579407685
2 1.2441588116708842
3 0.06106041923934216
4 0.049200446234917886
5 0.041479141222362015
6 0.036064678606653154
7 0.032064922221853316
8 0.02899328900843278
9 0.026561989758692364
10 0.024590092497778708
11 0.02295828148923116
12 0.02158479342596614
13 0.020411801650789874
14 0.019397313836791354
15 0.018510144089471034
16 0.017726680286239395
17 0.01702874155672761
18 0.016402120420266515
19 0.01583556770559199
